In [16]:
import ee

ee.Authenticate()
ee.Initialize(project='avilight-483312')

# --- Time range: January 2014 through December 2025 ---
START_DATE = ee.Date('2014-01-01')
N_MONTHS = 144

# --- Drive export ---
DRIVE_FOLDER = 'Satellite Datasets'
EXPORT_NAME = 'province_ndvi_monthly_2014_2025'

# --- Philippine provinces ---
# GAUL level 2 is the required province-level geography.
provinces = (
    ee.FeatureCollection('FAO/GAUL/2025/level2')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Philippines'))
)

print('Province features:', provinces.size().getInfo())


def mask_scale_ndvi(image):
    """
    Keep good MODIS pixels only, then convert raw integer NDVI
    to standard NDVI values.
    """
    qa_mask = image.select('SummaryQA').lte(1)

    return (
        image.select('NDVI')
        .updateMask(qa_mask)
        .multiply(0.0001)
        .rename('ndvi')
    )


def monthly_ndvi(month_index):
    """Create one quality-masked monthly NDVI image plus its valid-pixel band."""
    month_index = ee.Number(month_index)

    d0 = START_DATE.advance(month_index, 'month')
    d1 = d0.advance(1, 'month')

    monthly_collection = (
        ee.ImageCollection('MODIS/061/MOD13A1')
        .filterDate(d0, d1)
        .map(mask_scale_ndvi)
    )

    # Mean of the available 16-day composites within this calendar month.
    ndvi = monthly_collection.mean().rename('ndvi')

    # 1 where the monthly NDVI image has a valid value; masked otherwise.
    # Mean per province becomes the valid-pixel share.
    ndvi_valid = ndvi.mask().rename('ndvi_valid')

    return (
        ndvi.addBands(ndvi_valid)
        .set({
            'year': d0.get('year'),
            'month': d0.get('month'),
            'date': d0.format('YYYY-MM')
        })
    )


# Build 144 monthly images: 2014-01 through 2025-12.
month_indices = ee.List.sequence(0, N_MONTHS - 1)

ndvi_monthly = ee.ImageCollection.fromImages(
    month_indices.map(monthly_ndvi)
)


def reduce_month_to_provinces(image):
    """Calculate mean NDVI and valid-pixel share for every province."""
    reduced = image.reduceRegions(
        collection=provinces,
        reducer=ee.Reducer.mean(),
        scale=1000,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'adm1': feature.get('GAUL1_NAME'),
            'province': feature.get('GAUL2_NAME'),
            'adm2_code': feature.get('GAUL2_CODE'),
            'year': image.get('year'),
            'month': image.get('month'),
            'ndvi': feature.get('ndvi'),
            'ndvi_valid': feature.get('ndvi_valid')
        })

    return reduced.map(format_row)


# One row per province × month.
province_ndvi_fc = ndvi_monthly.map(
    reduce_month_to_provinces
).flatten()

# Export only the agreed columns.
province_ndvi_export = province_ndvi_fc.select([
    'adm1',
    'province',
    'adm2_code',
    'year',
    'month',
    'ndvi',
    'ndvi_valid'
])

task = ee.batch.Export.table.toDrive(
    collection=province_ndvi_export,
    description=EXPORT_NAME,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

task.start()

print('✅ Province-level monthly NDVI export started')
print('Task ID:', task.id)
print('Expected period: January 2014 to December 2025')
print('Drive folder:', DRIVE_FOLDER)

Province features: 85
✅ Province-level monthly NDVI export started
Task ID: 5JRNOWBHLKP3KQIEEWKXZXPN
Expected period: January 2014 to December 2025
Drive folder: Satellite Datasets


In [15]:
print(task.status())

{'state': 'COMPLETED', 'description': 'province_ndvi_monthly_2014_2025', 'priority': 100, 'creation_timestamp_ms': 1794715724889, 'update_timestamp_ms': 1794715845340, 'start_timestamp_ms': 1794715729058, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1-bU5d-UWd39u4OkyDb09gBu3DzCrayi9'], 'attempt': 1, 'batch_eecu_usage_seconds': 1526.8101806640625, 'id': 'ARSYXZ3ZNLXIO5TTIFMVBBLG', 'name': 'projects/avilight-483312/operations/ARSYXZ3ZNLXIO5TTIFMVBBLG'}


In [17]:
import pandas as pd

df = pd.read_csv('province_ndvi_monthly_2014_2025.csv')

df.loc[df['ndvi_valid'] < 0.3, 'ndvi'] = pd.NA

df.to_parquet('province_ndvi_monthly.parquet', index=False)

In [18]:
import pandas as pd

df_ndvi = pd.read_parquet('province_ndvi_monthly.parquet')

print('Total rows:', len(df_ndvi))
print('Provinces:', df_ndvi['adm2_code'].nunique())
print('Months:', df_ndvi[['year', 'month']].drop_duplicates().shape[0])
print('Expected rows:', df_ndvi['adm2_code'].nunique() * 144)

print('\nDate range:')
print(df_ndvi[['year', 'month']].drop_duplicates().sort_values(['year', 'month']).head())
print(df_ndvi[['year', 'month']].drop_duplicates().sort_values(['year', 'month']).tail())

print('\nRows with NDVI valid share below 0.3:')
print((df_ndvi['ndvi_valid'] < 0.3).sum())

Total rows: 12240
Provinces: 85
Months: 144
Expected rows: 12240

Date range:
     year  month
0    2014      1
85   2014      2
170  2014      3
255  2014      4
340  2014      5
       year  month
11815  2025      8
11900  2025      9
11985  2025     10
12070  2025     11
12155  2025     12

Rows with NDVI valid share below 0.3:
320


In [19]:
df_ndvi.loc[df_ndvi['ndvi_valid'] < 0.3, 'ndvi'] = pd.NA

df_ndvi.to_parquet('province_ndvi_monthly.parquet', index=False)

print('NDVI values set to missing:', df_ndvi['ndvi'].isna().sum())

NDVI values set to missing: 320
